# CME Futures: Portfolio Allocation

The baseline stage ranks complete configurations by equal-weight validation backtest Sharpe. For
each label, this notebook retains the strongest checkpoint and signal concentration for each of
the configured number of distinct model configurations, then evaluates the declared alternative
position sizing methods. Equal weight is not among them: it is the baseline itself, and because
`stage` is not part of `backtest_hash`, running it again here produces a row hashing
identically to its baseline parent, so one of the two is silently lost. Measured in this
case study's own pre-rebuild store: 48 rows stamped `stage='signal'` while carrying
`allocation.method='equal_weight'`, and no allocation-stage equal-weight rows at all.

All allocator lookbacks come from the case-study configuration. The official population is fixed
before execution; machine speed and caught failures cannot change which allocators run.

In [1]:
"""Run the declared CME futures allocation population."""

from case_studies.cme_futures.research_workflow import (
    ALL_LABELS,
    create_label_candidate_sets,
    open_study,
    product_universe_table,
    run_official_backtest_requests,
    shortlist_signal_configurations,
    strategy_request_frame,
)
from case_studies.utils.sweep_config import get_allocators, get_top_n_predictions

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str | None = None
PREVIEW_LABELS: list[str] = []
PREVIEW_MAX_BASELINE_ROWS = 0

## Select signal configurations by validation Sharpe

The shortlist is deterministic. It scans the immutable signal candidate set in descending Sharpe
order with the backtest identity as tie-break, and keeps one exact checkpoint and strategy per
distinct `(family, config_name)` pair.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE)
if EXECUTION_TIER == "canonical":
    if PREVIEW_LABELS or PREVIEW_MAX_BASELINE_ROWS:
        raise ValueError("canonical execution cannot declare preview reductions")
    labels = ALL_LABELS
elif EXECUTION_TIER == "preview":
    if WORKSPACE is None or not PREVIEW_LABELS or PREVIEW_MAX_BASELINE_ROWS < 1:
        raise ValueError(
            "preview execution requires WORKSPACE, PREVIEW_LABELS and PREVIEW_MAX_BASELINE_ROWS"
        )
    unknown = sorted(set(PREVIEW_LABELS) - set(ALL_LABELS))
    if unknown:
        raise ValueError(f"preview labels this case study does not declare: {unknown}")
    labels = tuple(PREVIEW_LABELS)
else:
    raise ValueError(f"unsupported execution tier: {EXECUTION_TIER!r}")
universe = product_universe_table()
universe

sector,product,expiry_rule,contract_months
str,str,str,str
"""agriculture""","""ZC""","""business_day_before_15th""","""H,K,N,U,Z"""
"""agriculture""","""ZL""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZM""","""business_day_before_15th""","""F,H,K,N,Q,U,V,Z"""
"""agriculture""","""ZS""","""business_day_before_15th""","""F,H,K,N,Q,U,X"""
"""agriculture""","""ZW""","""business_day_before_15th""","""H,K,N,U,Z"""
…,…,…,…
"""metals""","""SI""","""3rd_last_business_day""","""H,K,N,U,Z"""
"""treasuries""","""ZB""","""last_business_day""","""H,M,U,Z"""
"""treasuries""","""ZF""","""last_business_day""","""H,M,U,Z"""


## How many baseline configurations the position sizing methods run on

Canonical takes the shortlist size from `setup.yaml`, which is the declared width of the
allocation stage. A preview cannot: it backtests a bounded slice of the baseline stage, so the
canonical width names more distinct configurations than its pool contains and
`shortlist_signal_configurations` refuses - correctly, since silently returning fewer is the
quiet shrinking that strictness exists to prevent. The preview therefore declares its own
width, and is held to it just as strictly.

In [4]:
shortlist_size = (
    get_top_n_predictions("cme_futures", "allocation")
    if EXECUTION_TIER == "canonical"
    else PREVIEW_MAX_BASELINE_ROWS
)
allocators = get_allocators("cme_futures")
if not allocators:
    raise ValueError("the configured allocator population is empty")
if any(allocation.get("method") == "equal_weight" for allocation in allocators):
    raise ValueError(
        "equal_weight is the baseline stage, not an allocator: `stage` is not part of "
        "`backtest_hash`, so an equal-weight reweight hashes identically to its baseline "
        "parent and one of the two rows is lost. Remove it from the configured menu."
    )

request_rows = []
for label in labels:
    for baseline in shortlist_signal_configurations(
        study,
        label=label,
        limit=shortlist_size,
        execution_tier=EXECUTION_TIER,
    ):
        prediction_hash = baseline.registry_record()["prediction_hash"]
        signal = baseline.spec()["strategy"]["signal"]
        for allocation in allocators:
            method = allocation["method"]
            request_rows.append(
                {
                    "request_name": f"{baseline.hash}-{method}",
                    "prediction_hash": prediction_hash,
                    "label": label,
                    "signal": signal,
                    "allocation": allocation,
                    "risk": None,
                    "costs": None,
                    "chapter": "ch17",
                }
            )
requests = strategy_request_frame(request_rows)
requests.select("request_name", "prediction_hash", "label", "signal", "allocation")

request_name,prediction_hash,label,signal,allocation
str,str,str,object,object
"""6f36a8708d5b-score_weighted""","""b099266d758c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}",{'method': 'score_weighted'}
"""6f36a8708d5b-inverse_vol""","""b099266d758c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'inverse_vol', 'vol_window': 63}"
"""6f36a8708d5b-risk_parity""","""b099266d758c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'risk_parity', 'vol_window': 63}"
"""6f36a8708d5b-mvo_ledoit_wolf""","""b099266d758c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'mvo_ledoit_wolf', 'lookback': 63}"
"""6f36a8708d5b-hrp""","""b099266d758c""","""fwd_ret_5d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 5}","{'method': 'hrp', 'vol_window': 63}"
…,…,…,…,…
"""ab7b13f03bab-inverse_vol""","""fc591d94cc04""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'inverse_vol', 'vol_window': 63}"
"""ab7b13f03bab-risk_parity""","""fc591d94cc04""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'risk_parity', 'vol_window': 63}"
"""ab7b13f03bab-mvo_ledoit_wolf""","""fc591d94cc04""","""fwd_ret_21d""","{'long_short': True, 'method': 'equal_weight_top_k', 'top_k': 10}","{'method': 'mvo_ledoit_wolf', 'lookback': 63}"


## Execute and freeze allocation candidates

Moment-based allocators receive only price history before each decision. Product-keyed typed
decisions retain the selected prediction, roll audit, expiry reference, and allocation settings.

In [5]:
execution = run_official_backtest_requests(
    study,
    requests,
    population_name="cme_futures-allocation-validation-v1"
    if EXECUTION_TIER == "canonical"
    else None,
)
candidate_sets = (
    create_label_candidate_sets(study, execution, stage="allocation")
    if EXECUTION_TIER == "canonical"
    else {}
)

~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


...

~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


~/ml4t/public-s6-cme_futures/case_studies/utils/conformal.py:423: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  targets.join_asof(


`source` says whether each member was computed by this run or served from the registry because
an identical identity was already recorded. A re-run of a registered sweep is entirely `reused`
and completes in seconds; without the column that is indistinguishable from having computed
every row.

In [6]:
execution.catalog_rows.sort("label", "request_name")

request_name,label,prediction_hash,decision_hash,backtest_hash,complete,source
str,str,str,str,str,bool,str
"""7d19a3ea27b9-conformal_weighte…","""fwd_ret_21d""","""a3e9dd99182f""","""5fdc9d66b878""","""4f0f183be965""",true,"""computed"""
"""7d19a3ea27b9-hrp""","""fwd_ret_21d""","""a3e9dd99182f""","""0f6ff5634201""","""62090c5dde00""",true,"""computed"""
"""7d19a3ea27b9-inverse_vol""","""fwd_ret_21d""","""a3e9dd99182f""","""60dfcc4b7c2d""","""44f362f93225""",true,"""computed"""
"""7d19a3ea27b9-mvo_ledoit_wolf""","""fwd_ret_21d""","""a3e9dd99182f""","""4210e201bdb9""","""7f5a753bc399""",true,"""computed"""
"""7d19a3ea27b9-risk_parity""","""fwd_ret_21d""","""a3e9dd99182f""","""a4a782a89d12""","""b40c15e6e7dd""",true,"""computed"""
…,…,…,…,…,…,…
"""f8b8fcb0ff19-hrp""","""fwd_ret_5d""","""a3f6b9092f0f""","""ed05cfaeb049""","""da43932722bc""",true,"""computed"""
"""f8b8fcb0ff19-inverse_vol""","""fwd_ret_5d""","""a3f6b9092f0f""","""4f38911536db""","""e2178d05d8e7""",true,"""computed"""
"""f8b8fcb0ff19-mvo_ledoit_wolf""","""fwd_ret_5d""","""a3f6b9092f0f""","""c57b8a07d691""","""e27356bbbfd1""",true,"""computed"""


The next two execution notebooks select the highest validation Sharpe from the union of signal and
allocation results for each label. Cost sensitivity is diagnostic; risk overlays remain eligible
for final selection.